# Warehouse Activity Data Preprocessing Pipeline

**Purpose:** Clean and prepare warehouse activity logs for predictive modeling

**Input:**
- Activity logs (timestamped task completions)
- Location data (aisle/bay/level)
- Product data (weight/dimensions)
- Distance matrices (precomputed travel distances)

**Output:**
- Cleaned activity dataframe with time deltas and travel distances
- Exported to parquet for modeling

**Key Transformations:**
1. Remove assignment start events (non-productive time)
2. Calculate time between consecutive tasks per worker
3. Remove outlier gaps (breaks/errors) using 98th percentile
4. Join location/product attributes
5. Calculate travel distance between tasks

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Define workspace and warehouse parameters
warehouse_name = "OE"
base = "~/Lucas_Systems_Capstone_Project"

# Define data file paths and schema for loading
tables = {
    f"{warehouse_name}_Activity": f"{base}/data/database_backups_csv/{warehouse_name}/{warehouse_name}_Activity.csv",
    f"{warehouse_name}_Locations": f"{base}/data/database_backups_csv/{warehouse_name}/{warehouse_name}_Locations.csv",
    f"{warehouse_name}_Products": f"{base}/data/database_backups_csv/{warehouse_name}/{warehouse_name}_Products.csv",
}

column_names = {
    f"{warehouse_name}_Activity": ["ActivityCode","UserID","WorkCode","AssignmentID","ProductID","Quantity","Timestamp","LocationID"],
    f"{warehouse_name}_Locations": ["LocationID","Aisle","Bay","Level","Slot"],
    f"{warehouse_name}_Products": ["ProductID","ProductCode","UnitOfMeasure","Weight","Cube","Length","Width","Height"],
}

# Load tables into dictionary
dfs = {}
for name, fp in tables.items():
    dfs[name] = pd.read_csv(fp, header=0, names=column_names[name])

# Run this if there are no column names
#for name, fp in tables.items():
#    dfs[name] = pd.read_csv(fp, header=None, names=column_names[name])

# Load distance matrix
path = f"{base}/data/distance_matrices/distance_matrix_{warehouse_name}.csv"
Distance = pd.read_csv(path, index_col=0)
for c in Distance.columns:
    Distance[c] = pd.to_numeric(Distance[c], errors="coerce")


In [ ]:
# Display initial data summary and schema statistics
print("=" * 80)
for t in [f"{warehouse_name}_Activity", 
          f"{warehouse_name}_Locations", 
          f"{warehouse_name}_Products"]:

    print(f"Table: {t}")

    df = dfs[t]
    print(f"Dimensions: ({df.shape[0]} rows, {df.shape[1]} columns)\n")

    display(df.head(3))

    # Calculate data types and missing values
    schema_df = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "n_unique": df.nunique(dropna=True),
    })

    # Add numerical metrics
    num_df = df.select_dtypes(include="number")
    schema_df["min"] = num_df.min()
    schema_df["max"] = num_df.max()
    schema_df["mean"] = num_df.mean()

    display(schema_df)
    print("=" * 80)
    print("\n")

Table: OE_Activity
Dimensions: (96131 rows, 8 columns)



,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID
0,AssignmentOpen,64,10,7954429,NaN,NaN,2025-11-10 11:38:34.043,NaN
1,PickPut,419,20,7954541,6592.0,1.0,2025-11-10 11:39:42.330,14524.0
2,PickPut,419,20,7954542,6592.0,1.0,2025-11-10 11:39:42.883,14524.0


,dtype,n_missing,n_unique,min,max,mean
ActivityCode,object,0,2,NaN,NaN,NaN
UserID,int64,0,40,64.0,504.0,4.050260e+02
WorkCode,int64,0,3,10.0,30.0,2.642041e+01
AssignmentID,int64,0,42240,7717782.0,8042473.0,7.924902e+06
ProductID,float64,894,7691,1.0,57791.0,2.477799e+04
Quantity,float64,894,144,1.0,1143.0,8.453553e+00
Timestamp,object,0,95803,NaN,NaN,NaN
LocationID,float64,894,7669,1.0,8104198.0,8.932254e+05




Table: OE_Locations
Dimensions: (33518 rows, 5 columns)



,LocationID,Aisle,Bay,Level,Slot
0,2,40,5.0,4.0,4.0
1,3,40,9.0,1.0,2.0
2,4,42,21.0,6.0,3.0


,dtype,n_missing,n_unique,min,max,mean
LocationID,int64,0,33518,2.0,8034868.0,795298.495167
Aisle,object,0,52,NaN,NaN,NaN
Bay,float64,1,86,1.0,99.0,23.107080
Level,float64,1,12,1.0,50.0,3.687233
Slot,float64,1,35,1.0,35.0,3.019304




Table: OE_Products
Dimensions: (57670 rows, 8 columns)



,ProductID,ProductCode,UnitOfMeasure,Weight,Cube,Length,Width,Height
0,2,0204800418,BX,2.7338,0.263,NaN,NaN,NaN
1,3,07062B1322Q,EA,0.6500,0.027,NaN,NaN,NaN
2,4,07062B1324X,EA,2.4500,0.052,NaN,NaN,NaN


,dtype,n_missing,n_unique,min,max,mean
ProductID,int64,0,57670,2.0,57671.000,28836.500000
ProductCode,object,0,41981,NaN,NaN,NaN
UnitOfMeasure,object,0,55,NaN,NaN,NaN
Weight,float64,0,5572,0.0,7584.000,4.834325
Cube,float64,0,4009,0.0,421.296,0.623217
Length,float64,57670,0,NaN,NaN,NaN
Width,float64,57670,0,NaN,NaN,NaN
Height,float64,57670,0,NaN,NaN,NaN


In [ ]:
display(Distance.head())

# Convert Distance matrix to a long-format DataFrame for easier lookups
dist_long = (
    Distance.stack(dropna=False)
    .rename("distance")
    .reset_index()
    .rename(columns={"level_0": "FromLoc", "level_1": "ToLoc"})
)

display(dist_long.head())

,08|03|||,08|05|||,08|07|||,08|09|||,10|04|||,10|06|||,10|08|||,10|10|||,10|12|||,10|14|||,...,|Start L3,|Start L4,|Start L5,|Start L6,|Start R2,|Start R3,|Start R4,|Start R5,|Start R6,|Start SB
08|03|||,0,414,389,364,304,287,271,255,240,223,...,1094,1068,897,953,1080,1045,1089,913,969,994
08|05|||,25,0,414,389,329,312,296,280,265,248,...,1119,1093,922,978,1104,1070,1113,938,994,1018
08|07|||,50,25,0,414,354,337,321,305,290,273,...,1144,1118,947,1003,1129,1095,1138,963,1019,1043
08|09|||,75,50,25,0,379,362,346,330,315,298,...,1169,1143,972,1028,1154,1119,1163,988,1044,1068
10|04|||,484,459,434,410,0,19,35,51,67,81,...,1120,1074,958,838,929,1133,1087,971,854,767


/var/folders/r9/zx84vhy57q9_n8tk5dnw8g980000gn/T/ipykernel_84928/2095669890.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  Distance.stack(dropna=False)


,FromLoc,ToLoc,distance
0,08|03|||,08|03|||,0
1,08|03|||,08|05|||,414
2,08|03|||,08|07|||,389
3,08|03|||,08|09|||,364
4,08|03|||,10|04|||,304


In [ ]:
activity_key = f"{warehouse_name}_Activity"
locations_key = f"{warehouse_name}_Locations"
products_key = f"{warehouse_name}_Products"

# Clean activity data for merging
dfs[activity_key]["ProductID"] = pd.to_numeric(dfs[activity_key]["ProductID"], errors="coerce").astype("Int64")
dfs[activity_key]["Quantity"]  = pd.to_numeric(dfs[activity_key]["Quantity"], errors="coerce").astype("Int64")
dfs[activity_key]["LocationID"] = pd.to_numeric(dfs[activity_key]["LocationID"], errors="coerce").astype("Int64")
dfs[activity_key]["Timestamp"] = pd.to_datetime(dfs[activity_key]["Timestamp"], errors="coerce")
dfs[activity_key]["UserID"] = dfs[activity_key]["UserID"].astype(str)
dfs[activity_key]["WorkCode"] = dfs[activity_key]["WorkCode"].astype(str)
dfs[activity_key]["AssignmentID"] = dfs[activity_key]["AssignmentID"].astype(str)
dfs[activity_key] = dfs[activity_key].dropna(subset=["Timestamp"]).copy()

# Clean location data for merging
dfs[locations_key]["LocationID"] = pd.to_numeric(dfs[locations_key]["LocationID"], errors="coerce").astype("Int64")
for col in ["Bay", "Level", "Slot"]:
    dfs[locations_key][col] = pd.to_numeric(dfs[locations_key][col], errors="coerce").astype("Int64")

# Clean product data for merging
dfs[products_key]["ProductID"] = pd.to_numeric(dfs[products_key]["ProductID"], errors="coerce").astype("Int64")
dfs[products_key] = dfs[products_key][["ProductID", "ProductCode", "UnitOfMeasure", "Weight", "Cube"]]

Activity = dfs[activity_key]
Locations = dfs[locations_key]
Products = dfs[products_key]

In [ ]:
df_work = Activity.copy()
df_work = df_work.sort_values(["UserID", "Timestamp"]).reset_index(drop=True)

# Calculate time deltas between consecutive user tasks to identify task completion time
g = df_work.groupby("UserID", sort=False)
df_work["Prev_Timestamp"] = g["Timestamp"].shift(1)
df_work["Prev_LocationID"] = g["LocationID"].shift(1)

df_work["Time_Delta_sec"] = (
    df_work["Timestamp"] - df_work["Prev_Timestamp"]
).dt.total_seconds()

# Filter out gaps exceeding the 98th percentile (interpreting as breaks, chats, etc.)
threshold = df_work["Time_Delta_sec"].quantile(0.98)
print(f"98th Percentile Threshold: {threshold:.2f} seconds ({(threshold/60):.2f} minutes)")
df_work.loc[df_work["Time_Delta_sec"] > threshold, "Time_Delta_sec"] = np.nan
print(f"Number of rows with Time Deltas: {len(df_work[df_work['Time_Delta_sec'].notnull()])}")

Activity_prepped = df_work
display(Activity_prepped.head())

98th Percentile Threshold: 557.52 seconds (9.29 minutes)
Number of rows with Time Deltas: 94169


,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,Time_Delta_sec
0,PickPut,143,30,7717848,49658,160,2025-09-08 12:11:50.830,35192,NaT,<NA>,NaN
1,PickPut,143,30,7717860,460,50,2025-09-08 12:12:18.127,422,2025-09-08 12:11:50.830,35192,27.297
2,PickPut,143,30,7717908,460,100,2025-09-08 12:15:46.650,422,2025-09-08 12:12:18.127,422,208.523
3,PickPut,143,30,7717921,44547,13,2025-09-08 12:16:30.470,10743,2025-09-08 12:15:46.650,422,43.820
4,PickPut,143,30,7717920,44547,13,2025-09-08 12:18:00.970,10743,2025-09-08 12:16:30.470,10743,90.500


In [ ]:
# Join Activity data with Product and Location features
df_joined = Activity_prepped.merge(Products, on="ProductID", how="left")
df_joined = df_joined.merge(Locations, on="LocationID", how="left")

df_joined = df_joined.merge(
    Locations[["LocationID","Aisle","Bay","Level","Slot"]].rename(columns={
        "LocationID": "Prev_LocationID",
        "Aisle": "Prev_Aisle",
        "Bay": "Prev_Bay",
        "Level": "Prev_Level",
        "Slot": "Prev_Slot",
    }),
    on="Prev_LocationID",
    how="left"
)

display(df_joined.head())

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Weight,Cube,Aisle,Bay,Level,Slot,Prev_Aisle,Prev_Bay,Prev_Level,Prev_Slot
0,PickPut,143,30,7717848,49658,160,2025-09-08 12:11:50.830,35192,NaT,<NA>,...,0.0113,0.005,40,19,2,2,NaN,<NA>,<NA>,<NA>
1,PickPut,143,30,7717860,460,50,2025-09-08 12:12:18.127,422,2025-09-08 12:11:50.830,35192,...,0.0300,0.016,40,18,2,1,40,19,2,2
2,PickPut,143,30,7717908,460,100,2025-09-08 12:15:46.650,422,2025-09-08 12:12:18.127,422,...,0.0300,0.016,40,18,2,1,40,18,2,1
3,PickPut,143,30,7717921,44547,13,2025-09-08 12:16:30.470,10743,2025-09-08 12:15:46.650,422,...,0.5000,0.139,40,18,2,2,40,18,2,1
4,PickPut,143,30,7717920,44547,13,2025-09-08 12:18:00.970,10743,2025-09-08 12:16:30.470,10743,...,0.5000,0.139,40,18,2,2,40,18,2,2


In [ ]:
df_detailed = df_joined.copy()

df_detailed["Aisle2"] = pd.to_numeric(df_detailed["Aisle"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
df_detailed["Bay2"] = pd.to_numeric(df_detailed["Bay"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
df_detailed["Prev_Aisle2"] = pd.to_numeric(df_detailed["Prev_Aisle"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
df_detailed["Prev_Bay2"] = pd.to_numeric(df_detailed["Prev_Bay"], errors="coerce").astype("Int64").astype(str).str.zfill(2)

# Create location keys and map travel distances
df_detailed["LocKey"] = df_detailed["Aisle2"] + "|" + df_detailed["Bay2"] + "|||"
df_detailed["PrevLocKey"] = df_detailed["Prev_Aisle2"] + "|" + df_detailed["Prev_Bay2"] + "|||"

# Calculate travel distances using the distance matrix (long format)
df_detailed = df_detailed.merge(
    dist_long,
    left_on=["LocKey", "PrevLocKey"],
    right_on=["FromLoc", "ToLoc"],
    how="left"
).rename(columns={"distance": "Travel_Distance"}).drop(columns=["FromLoc", "ToLoc"])

display(df_detailed.head())


,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Prev_Bay,Prev_Level,Prev_Slot,Aisle2,Bay2,Prev_Aisle2,Prev_Bay2,LocKey,PrevLocKey,Travel_Distance
0,PickPut,143,30,7717848,49658,160,2025-09-08 12:11:50.830,35192,NaT,<NA>,...,<NA>,<NA>,<NA>,40,19,<NA>,<NA>,40|19|||,<NA>|<NA>|||,NaN
1,PickPut,143,30,7717860,460,50,2025-09-08 12:12:18.127,422,2025-09-08 12:11:50.830,35192,...,19,2,2,40,18,40,19,40|18|||,40|19|||,21.0
2,PickPut,143,30,7717908,460,100,2025-09-08 12:15:46.650,422,2025-09-08 12:12:18.127,422,...,18,2,1,40,18,40,18,40|18|||,40|18|||,0.0
3,PickPut,143,30,7717921,44547,13,2025-09-08 12:16:30.470,10743,2025-09-08 12:15:46.650,422,...,18,2,1,40,18,40,18,40|18|||,40|18|||,0.0
4,PickPut,143,30,7717920,44547,13,2025-09-08 12:18:00.970,10743,2025-09-08 12:16:30.470,10743,...,18,2,2,40,18,40,18,40|18|||,40|18|||,0.0


In [ ]:
# Drop assignment-opening noise to refine dataset to pure pick task cycles
open_indices = df_detailed[df_detailed["ActivityCode"] == "AssignmentOpen"].index
first_activity_indices = open_indices + 1
to_drop = open_indices.union(first_activity_indices).intersection(df_detailed.index)

# Create the final cleaned dataset
Detailed_Data = df_detailed.drop(to_drop).reset_index(drop=True)

In [ ]:
# Export processed files
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

Detailed_Data.to_parquet(output_dir / f"{warehouse_name.lower()}_detailed.parquet", index=False)
Activity_prepped.to_parquet(output_dir / f"{warehouse_name.lower()}_activity_prepped.parquet", index=False)
df_joined.to_parquet(output_dir / f"{warehouse_name.lower()}_joined.parquet", index=False)

print(f"Successfully exported all {warehouse_name} files to {output_dir}")

Successfully exported all OE files to /Users/betsyfrdmn/Lucas_Systems_Capstone_Project/data/processed


# Extra

In [ ]:
df = Activity_prepped.copy()
df = df.dropna(subset=["ProductID", "Time_Delta_sec"]).copy()

df["Prev_ProductID"] = df.groupby("UserID")["ProductID"].shift(1)
df_pairs = df[df["ProductID"] == df["Prev_ProductID"]].copy()

product_pick_times = (
    df_pairs.groupby("ProductID")
            .agg(
                n_pairs=("Time_Delta_sec", "size"),
                avg_pick_time_sec=("Time_Delta_sec", "mean"),
                median_pick_time_sec=("Time_Delta_sec", "median"),
                std_pick_time_sec=("Time_Delta_sec", "std")
            )
            .reset_index()
            .sort_values("ProductID")
)

# display(product_pick_times.head())
# product_pick_times.to_csv(f"{base}/data/processed/product_pick_times.csv", index=False)

,ProductID,n_pairs,avg_pick_time_sec,median_pick_time_sec,std_pick_time_sec
0,1,281,3.867434,0.2800,34.893334
1,3,94,15.717170,9.3815,16.637097
2,4,37,17.089459,17.6200,12.933894
3,7,4,18.214000,4.3960,29.545528
4,8,2,21.388500,21.3885,29.337153
